In [1]:
# # Install dependencies for Unsloth + GPT-OSS
# !pip install --upgrade -qqq uv
# !uv pip install -qqq \
#     "torch>=2.8.0" "triton>=3.4.0" numpy pillow torchvision bitsandbytes "transformers==4.56.2" \
#     "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#     "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#     git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
# !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# # Install openai_harmony (Harmony protocol tools)
# !pip install -q openai-harmony jupyter_client pandas datasets


In [2]:
class CONFIG:
    TRAIN_SIZE = 10_000
    BATCH_SIZE = 1
    EPOCHS = 1 
    LEARNING_RATE = 1e-4
    MAX_SEQ_LENGTH = 60_000 #4096*2
    MODEL_PATH: str | None = None
    KAGGLE=True 
    MASK_THINK = True
    KEEP_UNIQUE = False
    SEED = 42
    REASONING_EFFORT = "high"
    SAVE_STEPS = 100
    SAVE_TOTAL_LIMIT = 2
    RESUME_FROM_CHECKPOINT = False
    CHECKPOINT_PATH = None
    MERGE_INPUT_FILES = False
    DEFAULT_INPUT_FILE = "/kaggle/input/datasets/barnobarno/nemotron-high-reasoning-pass-1-and-2-and-3-and-4/High_low_pass.jsonl"
    MERGE_INPUT_FILES_LIST = [
        "/kaggle/input/datasets/barnobarno/nemotron-high-reasoning-pass-1-and-2-and-3-and-4/High_low_pass.jsonl",
        "/kaggle/input/datasets/barnobarno/nemotron-high-reasoning-pass-1-and-2-and-3-and-4/High_medium_pass.jsonl",
    ]
    

cfg = CONFIG()
cfg.MODEL_PATH = "/kaggle/input/models/barnobarno/gpt-oss-120b-bnb-4bit/transformers/unsloth/1" if cfg.KAGGLE else "unsloth/gpt-oss-20b"

In [3]:
!python --version

Python 3.12.12


In [4]:
print("STARTING THE AHHHHHHHHHHHHHHHHHHHHHHHHHHHH")

STARTING THE AHHHHHHHHHHHHHHHHHHHHHHHHHHHH


In [5]:
if cfg.KAGGLE:
    !uv pip install --system --no-index --find-links='/kaggle/input/datasets/barnobarno/unsloth-py-3-12/unsloth' 'unsloth'


Using Python 3.12.12 environment at: /usr
Resolved 86 packages in 364ms
Prepared 27 packages in 31.14s
Uninstalled 19 packages in 2.70s
Installed 27 packages in 161ms
 + bitsandbytes==0.49.1
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.3.0
 - huggingface-hub==1.4.1
 + huggingface-hub==0.36.0
 + msgspec==0.20.0
 - nvidia-cublas-cu12==12.6.4.1
 + nvidia-cublas-cu12==12.8.4.1
 - nvidia-cuda-cupti-cu12==12.6.80
 + nvidia-cuda-cupti-cu12==12.8.90
 - nvidia-cuda-nvrtc-cu12==12.6.77
 + nvidia-cuda-nvrtc-cu12==12.8.93
 - nvidia-cuda-runtime-cu12==12.6.77
 + nvidia-cuda-runtime-cu12==12.8.90
 - nvidia-cufft-cu12==11.3.0.4
 + nvidia-cufft-cu12==11.3.3.83
 - nvidia-cufile-cu12==1.11.1.6
 + nvidia-cufile-cu12==1.13.1.3
 - nvidia-curand-cu12==10.3.7.77
 + nvidia-curand-cu12==10.3.9.90
 - nvidia-cusolver-cu12==11.7.1.2
 + nvidia-cusolver-cu12==11.7.3.90
 - nvidia-cusparse-cu12==12.5.4.2
 + nvidia-cusparse-cu12==12.5.8.93
 - nvidia-nvjitlink-cu12==12.6.85
 + nvidia-nvjitlink-cu12==1

In [6]:
try:
    from unsloth import FastLanguageModel
except:
    !uv pip install --system --no-index --find-links='/kaggle/input/datasets/barnobarno/unsloth-library/unsloth' 'unsloth'
    from unsloth import FastLanguageModel

    

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-02-19 22:19:30.047295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771539570.383178      44 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771539570.479974      44 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771539571.316231      44 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771539571.316259      44 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771539571.316262      44 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
local_files_only = True if cfg.KAGGLE else False

In [8]:
# del model
# del tokenizer

In [9]:
import torch
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = cfg.MODEL_PATH ,
    dtype = dtype, # None for auto detection
    max_seq_length = cfg.MAX_SEQ_LENGTH, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    local_files_only = local_files_only
)

==((====))==  Unsloth 2026.1.3: Fast Gpt_Oss patching. Transformers: 4.57.3.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/16 [00:00<?, ?it/s]

In [10]:
import polars as pl

files = cfg.MERGE_INPUT_FILES_LIST if cfg.MERGE_INPUT_FILES else [cfg.DEFAULT_INPUT_FILE]
print(f"MERGE_INPUT_FILES={cfg.MERGE_INPUT_FILES} | Files: {files}")

def load_all_columns_lazy(file_path):
    # 1. infer_schema_length=None forces it to scan ALL rows to find sparse columns like 'tools'
    lazy_df = pl.scan_ndjson(file_path, infer_schema_length=None, ignore_errors=True)
    
    # 2. "Reverse DropNA" Logic
    # We perform the filter HERE (lazily) instead of after loading.
    # This gives you the same result as "filtering later" but saves the RAM 
    # that would have been wasted loading the 'tools' rows.
    schema_keys = lazy_df.collect_schema().names()
    
    if "tools" in schema_keys:
        lazy_df = lazy_df.filter(pl.col("tools").is_null())
    
    # 3. REMOVED .select() -> Now returns ALL columns
    return lazy_df

# Setup scans
queries = [load_all_columns_lazy(file_path) for file_path in files]

# Collect
print("Scanning and filtering (keeping all columns)...")

# CRITICAL: how="diagonal" ensures that if 'metadata' or 'url' is missing 
# in one file but present in the other, it doesn't crash.
df = pl.concat(queries, how="diagonal").collect()

print(f"Loaded {len(df)} rows with columns: {df.columns}")
print(df.head())

MERGE_INPUT_FILES=False | Files: ['/kaggle/input/datasets/barnobarno/nemotron-high-reasoning-pass-1-and-2-and-3-and-4/High_low_pass.jsonl']
Scanning and filtering (keeping all columns)...
Loaded 135098 rows with columns: ['expected_answer', 'problem', 'original_expected_answer', 'changed_answer_to_majority', 'data_source', 'messages', 'tools', 'used_in', 'metadata', 'license', 'url', 'user_url', 'user_name']
shape: (5, 13)
┌─────────────┬────────────┬────────────┬────────────┬───┬───────────┬──────┬──────────┬───────────┐
│ expected_an ┆ problem    ┆ original_e ┆ changed_an ┆ … ┆ license   ┆ url  ┆ user_url ┆ user_name │
│ swer        ┆ ---        ┆ xpected_an ┆ swer_to_ma ┆   ┆ ---       ┆ ---  ┆ ---      ┆ ---       │
│ ---         ┆ str        ┆ swer       ┆ jority     ┆   ┆ str       ┆ str  ┆ str      ┆ str       │
│ str         ┆            ┆ ---        ┆ ---        ┆   ┆           ┆      ┆          ┆           │
│             ┆            ┆ str        ┆ bool       ┆   ┆          

In [11]:
import polars as pl
import random
import numpy as np
import torch

# --- CONFIGURATION ---
KEEP_INT_ONLY = True
CHANGE_SYSTEM_PROMPT = False 

# Used only when CHANGE_SYSTEM_PROMPT = True
NEW_SYSTEM_PROMPT = (
    "You are an expert Math Olympiad solver. Your goal is to solve complex "
    "mathematical problems with rigorous, step-by-step reasoning.\n"
    "Knowledge cutoff: 2026-01\n"
    "Current date: 2026-01-22\n\n"
    "Reasoning: medium\n\n"
    "# Valid channels: analysis, final. Channel must be included for every message."
)

# --- 0. REPRODUCIBILITY ---
random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)

# --- 1. FILTERING ---
dataset = df.filter(
    pl.col("tools").is_null()
).drop(
    ["uuid", "original_expected_answer", "license", "used_in", "user_name", "user_url", "url", "tools"], 
    strict=False
)

# Integer Filter
if KEEP_INT_ONLY:
    dataset = dataset.with_columns(
        pl.col("expected_answer")
        .str.extract(r"(-?\d+)", 1)
        .cast(pl.Int64, strict=False)
        .alias("numeric_value")
    ).filter(
        pl.col("numeric_value").is_not_null()
    )

# Optional uniqueness control
if cfg.KEEP_UNIQUE:
    unique_key = "problem" if "problem" in dataset.columns else "expected_answer"
    before_unique = len(dataset)
    dataset = dataset.unique(subset=[unique_key], keep="first")
    print(f"KEEP_UNIQUE=True | key={unique_key} | {before_unique} -> {len(dataset)}")

# Pre-sampling for speed
PRE_SAMPLE_SIZE = int(cfg.TRAIN_SIZE * 1.5)
if len(dataset) > PRE_SAMPLE_SIZE:
    dataset = dataset.sample(n=PRE_SAMPLE_SIZE, seed=cfg.SEED, shuffle=True)

# Enforce exact train size target
if len(dataset) > cfg.TRAIN_SIZE:
    dataset = dataset.sample(n=cfg.TRAIN_SIZE, seed=cfg.SEED, shuffle=True)
elif len(dataset) < cfg.TRAIN_SIZE:
    print(f"⚠️ Requested TRAIN_SIZE={cfg.TRAIN_SIZE}, available rows={len(dataset)}. Using available rows.")

# --- 2. CLEANUP ---
def clean_messages(messages):
    if messages is None:
        return []
    
    new_history = []
    
    for msg in messages:
        new_msg = dict(msg)
        
        # Skip Tool Roles
        if new_msg.get('role') == 'tool':
            continue
            
        # Clean artifacts
        new_msg.pop('tool_calls', None)
        new_msg.pop('tool_call_id', None)

        if new_msg.get('reasoning_content'):
            new_msg['thinking'] = new_msg.pop('reasoning_content')
            
        new_msg = {k: v for k, v in new_msg.items() if v is not None}
        new_history.append(new_msg)
        
    return new_history

print("Cleaning messages...")
dataset = dataset.with_columns(
    pl.col("messages").map_elements(clean_messages, return_dtype=pl.Object).alias("AA")
)

# --- 3. TOKENIZE (+ OPTIONAL SYSTEM PROMPT SWAP) ---
def replace_system_prompt_block(text, new_prompt):
    start_tag = "<|start|>system<|message|>"
    end_tag = "<|end|>"

    original_count = text.count(start_tag)
    if original_count == 0:
        return f"{start_tag}{new_prompt}{end_tag}" + text

    start_idx = text.find(start_tag)
    content_start = start_idx + len(start_tag)
    end_idx = text.find(end_tag, content_start)
    if end_idx == -1:
        raise ValueError("Malformed system block: missing <|end|> after system start tag.")

    updated = text[:content_start] + new_prompt + text[end_idx:]

    # Safety: do not change number of system tags; only replace first block content
    if updated.count(start_tag) != original_count:
        raise ValueError("System prompt replacement changed system tag count unexpectedly.")

    if new_prompt not in updated[content_start:content_start + len(new_prompt) + 4]:
        raise ValueError("System prompt replacement failed to place new prompt in first system block.")

    return updated

def apply_template_and_maybe_swap(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        reasoning_effort=cfg.REASONING_EFFORT
    )

    # Normal behavior path (same idea as low-training notebook): no prompt rewriting
    if not CHANGE_SYSTEM_PROMPT:
        return text

    # Optional custom system prompt path (robust first-system-block replacement)
    return replace_system_prompt_block(text, NEW_SYSTEM_PROMPT)

print(f"Tokenizing (System Prompt Swap: {CHANGE_SYSTEM_PROMPT})...")
dataset = dataset.with_columns(
    pl.col("AA").map_elements(apply_template_and_maybe_swap, return_dtype=pl.String).alias("text")
)

# --- 4. VERIFY ---
print("\n--- Final Prompt Check (First 500 chars) ---")
print(dataset["text"][0][:500])

if "developer" in dataset["text"][0][:500]:
    print("⚠️ WARNING: 'developer' role still found.")
else:
    print("✅ 'developer' role gone. System prompt is clean.")

print(f"Final dataset rows: {len(dataset)}")

Cleaning messages...
Tokenizing (System Prompt Swap: False)...

--- Final Prompt Check (First 500 chars) ---
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-02-19

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve the following math problem. Make sure to put the answer (and only answer) inside \boxed{}.

Given a concave function \( f(x) \), which is larger: \( (
✅ 'developer' role gone. System prompt is clean.
Final dataset rows: 10000


In [12]:
# Show answers that failed the strict number cast
dropped_rows = df.filter(
    pl.col("tools").is_null()
).filter(
    pl.col("expected_answer").cast(pl.Float64, strict=False).is_null()
).select(["expected_answer"]).head(20)

print("--- REJECTED ANSWERS (Sample) ---")
print(dropped_rows)

--- REJECTED ANSWERS (Sample) ---
shape: (20, 1)
┌─────────────────────────────────┐
│ expected_answer                 │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ \( 3^{13} - 3 \)                │
│ \;y\bigl(\sqrt{x^{2}+y^{2}}+x\… │
│ 30\;\text{by}\;27               │
│ \(\frac{2abc}{ab + bc + ca}\)   │
│ a = b                           │
│ …                               │
│ a=8                             │
│ \( a = 1.465, b = \frac{\pi}{6… │
│ \[                              │
│ x = \frac{S^2 + b^2 + c^2 -…    │
│ y=x+1                           │
│ $(-\infty ;4]\cup \left[\frac{… │
└─────────────────────────────────┘


In [13]:
# dataset["text"][0]

In [14]:
# from datasets import Dataset

# # 1. Pre-processing
# train_data = df.copy()
# train_data.drop(columns=["uuid", "original_expected_answer", "license", "used_in", "user_name", "user_url", "url"], inplace=True, axis=1)

# # Filter out rows with tools (as per your snippet)
# train_data = train_data[train_data["tools"].isna()]
# train_data.drop(columns=["tools"], inplace=True, axis=1)

# def remove_none_keys(messages):
#     return [{k: v for k, v in entry.items() if v is not None} for entry in messages]

# def format_for_gpt_oss(example):
#     messages = example['messages']
    
#     # --- MODIFICATION START ---
#     # Initialize with the System Prompt
#     new_messages = [{
#         "role": "system", 
#         "content": "YOU ARE A MATH EXPERT"
#     }]
#     # --- MODIFICATION END ---
    
#     for msg in messages:
#         # Optional: Skip existing system prompts to strictly enforce your new one
#         if msg.get('role') == 'system':
#             continue

#         new_msg = msg.copy()
        
#         # 1. Rename 'reasoning_content' to 'thinking'
#         if 'reasoning_content' in new_msg:
#             new_msg['thinking'] = new_msg.pop('reasoning_content')
        
#         # 2. Ensure intermediate tool steps don't conflict
#         if new_msg.get('tool_calls') and new_msg.get('thinking'):
#              new_msg['content'] = "" 

#         new_messages.append(new_msg)
    
#     return {'messages': new_messages}

# # Apply to your dataset
# train_data_formatted = train_data.apply(format_for_gpt_oss, axis=1)
# train_data["messages"] = train_data_formatted

# # Extract the list of messages
# train_data["AA"] = train_data["messages"].apply(lambda x: x["messages"])

# # Create the final dataset slice
# dataset = train_data.iloc[0:cfg.TRAIN_SIZE].copy()
# dataset["AA"] = dataset["AA"].apply(remove_none_keys)

# # Apply Chat Template
# dataset["text"] = dataset.apply(lambda row: tokenizer.apply_chat_template(
#     row["AA"], 
#     tokenize=False, 
#     add_generation_prompt=True,
#     reasoning_effort="low"
# ), axis=1)

In [15]:
# Add LoRA adapters with rank 16
from datasets import Dataset

# hf_dataset = Dataset.from_pandas(dataset)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [16]:
import gc
import torch
gc.collect()


0

In [17]:
import pyarrow as pa
from datasets import Dataset

print("🔄 Converting Polars DataFrame to Hugging Face Dataset...")

# 1. Select only the columns needed for training to save RAM
# SFTTrainer strictly needs 'text'. We keep 'expected_answer' just for reference/debug if needed.
# If your dataset is huge, you can just do .select(["text"])
hf_arrow_table = dataset.select(["text"]).to_arrow()

# 2. Create the Hugging Face Dataset directly from Arrow (Zero-Copy)
hf_dataset = Dataset(hf_arrow_table)

# 3. Validation
print(f"✅ Hugging Face Dataset created successfully!")
print(f"Shape: {hf_dataset.shape}")
print(f"Column Names: {hf_dataset.column_names}")

# Verify the text field is correct for SFTTrainer
print("\n--- Sample Training Entry ---")
print(hf_dataset[0]["text"][:500]) # Print first 500 chars

🔄 Converting Polars DataFrame to Hugging Face Dataset...
✅ Hugging Face Dataset created successfully!
Shape: (10000, 1)
Column Names: ['text']

--- Sample Training Entry ---
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-02-19

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve the following math problem. Make sure to put the answer (and only answer) inside \boxed{}.

Given a concave function \( f(x) \), which is larger: \( (


In [18]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=cfg.MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=cfg.BATCH_SIZE,
        gradient_accumulation_steps=2,
        warmup_steps=5,
        #num_train_epochs=cfg.EPOCHS,
        max_steps=20,
        learning_rate=cfg.LEARNING_RATE,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=cfg.SEED,
        output_dir="outputs",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        group_by_length=True,
        save_strategy="steps",
        save_steps=cfg.SAVE_STEPS,
        save_total_limit=cfg.SAVE_TOTAL_LIMIT,
    ),
)



Unsloth: Tokenizing ["text"] (num_proc=30):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [19]:
# CHANGE THIS: Point to the analysis channel instead of the final channel sometime
if not cfg.MASK_THINK:
    try:
        gpt_oss_kwargs = dict(
            instruction_part = "<|start|>user<|message|>", 
            response_part = "<|start|>assistant<|channel|>analysis<|message|>"
        )
        
        trainer = train_on_responses_only(
            trainer,
            **gpt_oss_kwargs,
        )
        TRAIN=True
    except:
        TRAIN = False
else:
    gpt_oss_kwargs = dict(
    instruction_part="<|start|>user<|message|>", 
    response_part="<|start|>assistant<|channel|>final<|message|>"
)
    trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)
    TRAIN = True



    


print(f"TRAINING READY AS NOT MASKING THE THINKING FROM TRAIN LOSS {cfg.MASK_THINK}")

Map (num_proc=30):   0%|          | 0/10000 [00:00<?, ? examples/s]

TRAINING READY AS NOT MASKING THE THINKING FROM TRAIN LOSS True


In [20]:
import os

# 1. Resolve optional resume checkpoint
resume_checkpoint = None
if cfg.RESUME_FROM_CHECKPOINT:
    if cfg.CHECKPOINT_PATH:
        resume_checkpoint = cfg.CHECKPOINT_PATH
    else:
        output_dir = "outputs"
        if os.path.isdir(output_dir):
            checkpoints = [
                directory
                for directory in os.listdir(output_dir)
                if directory.startswith("checkpoint-") and os.path.isdir(os.path.join(output_dir, directory))
            ]
            if checkpoints:
                checkpoints = sorted(
                    checkpoints,
                    key=lambda name: int(name.split("-")[-1]) if name.split("-")[-1].isdigit() else -1
                )
                resume_checkpoint = os.path.join(output_dir, checkpoints[-1])

    if resume_checkpoint:
        print(f"Resuming training from: {resume_checkpoint}")
    else:
        print("⚠️ RESUME_FROM_CHECKPOINT=True but no checkpoint found. Starting fresh.")

# 2. Train the model
train_time_seconds = 0.0
if TRAIN:
    if resume_checkpoint:
        trainer_stats = trainer.train(resume_from_checkpoint=resume_checkpoint)
    else:
        trainer_stats = trainer.train()
    train_time_seconds = float(trainer_stats.metrics.get("train_runtime", 0.0))

# 3. Memory stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

# 4. Time stats
train_time_minutes = round(train_time_seconds / 60, 2)

print(f"Peak reserved memory = {used_memory} GB")
print(f"Total training time  = {train_time_minutes} minutes ({train_time_seconds:.2f} seconds)")

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 23,887,872 of 116,853,044,544 (0.02% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.286300
2,0.407600
3,0.240300
4,0.287200
5,0.234300
6,0.219700
7,0.169900
8,0.241500
9,0.232500
10,0.223000


Peak reserved memory = 74.518 GB
Total training time  = 12.49 minutes (749.37 seconds)


In [21]:
model.save_pretrained("gpt_oss_20b_nemotronv2_HIGH_FIXED")
tokenizer.save_pretrained("gpt_oss_20b_nemotronv2_HIGH_FIXED")
print("Model saved to 'gpt_oss_20b_nemotronv2_HIGH_FIXED'")

Model saved to 'gpt_oss_20b_nemotronv2_HIGH_FIXED'
